In [1]:
#Use Tweepy to Authenticate with Twitter

In [2]:
import tweepy, keys

In [3]:
#Authenticate with Twitter

In [4]:
auth = tweepy.OAuthHandler(keys.consumer_key, keys.consumer_secret)
auth.set_access_token(keys.access_token, keys.access_token_secret)

In [5]:
#Configure the Tweepy API Object to wait if our app reaches any Twitter rate limits

In [6]:
api = tweepy.API(auth, wait_on_rate_limit=True, wait_on_rate_limit_notify=True)

In [7]:
import pandas as pd

In [8]:
senators_df = pd.read_csv('senators.csv')

In [9]:
senators_df['TwitterID'] = senators_df['TwitterID'].astype(str)

In [10]:
pd.options.display.max_columns=6            #Showing max 6 columns

In [11]:
senators_df.head()

,State,Name,Party,TwitterHandle,TwitterID
0,AL,Richard Shelby,R,SenShelby,21111098
1,AL,Doug Jomes,D,SenDougJones,941080085121175552
2,AK,Lisa Murkowski,R,lisamurkowski,18061669
3,AK,Dan Sullivan,R,SenDanSullivan,2891210047
4,AZ,Jon Kyl,R,SenJonKyl,24905240


In [12]:
#Configuring the MongoClient

In [13]:
from pymongo import MongoClient

In [14]:
atlas_client = MongoClient(keys.mongo_connection_string)            #Connect to MongoDB Atlas cluster

In [15]:
db = atlas_client.senators                                          #Creates the database if it doesn't exist

In [16]:
#Setting up Tweet Stream

In [17]:
from tweetlistener import TweetListener

In [18]:
tweet_limit = 10000

In [19]:
twitter_stream = tweepy.Stream(api.auth, TweetListener(api, db, tweet_limit))

In [20]:
#Starting the Tweet Stream

In [21]:
#twitter_stream.filter(track=senators_df.TwitterHandle.tolist(), follow=senators_df.TwitterID.tolist())

In [22]:
db.tweets.create_index([('$**', 'text')])               #Create a text index. $** indicates that every text field in a document should be indexed for a full-text search

'$**_text'

In [23]:
tweet_counts = []
for senator in senators_df.TwitterHandle:
    tweet_counts.append(db.tweets.count_documents(              #Count the total number of documents in the collection that contain the specified text
        {"$text":{"$search": senator}}                          #We're using the index named text to search for the value of senator
    ))

In [24]:
tweet_counts_df = senators_df.assign(Tweets=tweet_counts)

In [25]:
tweet_counts_df.sort_values(by='Tweets', ascending=False).head(10)

,State,Name,Party,TwitterHandle,TwitterID,Tweets
0,AL,Richard Shelby,R,SenShelby,21111098,0
1,AL,Doug Jomes,D,SenDougJones,941080085121175552,0
2,AK,Lisa Murkowski,R,lisamurkowski,18061669,0
3,AK,Dan Sullivan,R,SenDanSullivan,2891210047,0
4,AZ,Jon Kyl,R,SenJonKyl,24905240,0
5,AZ,Jeff Flake,R,JeffFlake,16056306,0
6,AR,Tom Cotton,R,SenTomCotton,968650362,0
7,AR,John Boozman,R,JohnBoozman,5558312,0
8,CA,Dianne Feinstein,D,SenFeinstein,476256944,0
9,CA,Kamala Harris,D,SenKamalaHarris,803694179079458816,0


In [26]:
from geopy import OpenMapQuest

In [27]:
import time

In [28]:
from state_codes import state_codes

In [29]:
#Let's get the geocoder object to translate location names into Location objects

In [30]:
geo = OpenMapQuest(api_key=keys.mapquest_key)

In [31]:
states = tweet_counts_df.State.unique()

In [32]:
states

<StringArray>
['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL',
 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS', 'MO', 'MT',
 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI',
 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY']
Length: 50, dtype: str

In [33]:
locations = []

In [34]:
for state in states:
    processed = False
    delay = .1
    while not processed:
        try:
            locations.append(geo.geocode(state_codes[state] + ', USA'))
            print(locations[-1])
            processed = True
        except:     #Timed out, so wait before trying again
            print('OpenMapQuest service timed out. Waiting')
            time.sleep(delay)
            delay += .1

OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting
OpenMapQuest service timed out. Waiting


KeyboardInterrupt: 

In [35]:
#Grouping the Tweet Counts by State

In [36]:
tweets_counts_by_state = tweet_counts_df.groupby(
    'State',
    as_index=False              #Indicates that the state codes shoudl be values in a column of the sulting GroupBy object, rather than the indices for the wors
).sum()                         #Totals the numeric data

In [37]:
tweets_counts_by_state.head()

,State,Name,Party,TwitterHandle,TwitterID,Tweets
0,AK,Lisa MurkowskiDan Sullivan,RR,lisamurkowskiSenDanSullivan,180616692891210047,0
1,AL,Richard ShelbyDoug Jomes,RD,SenShelbySenDougJones,21111098941080085121175552,0
2,AR,Tom CottonJohn Boozman,RR,SenTomCottonJohnBoozman,9686503625558312,0
3,AZ,Jon KylJeff Flake,RR,SenJonKylJeffFlake,2490524016056306,0
4,CA,Dianne FeinsteinKamala Harris,DD,SenFeinsteinSenKamalaHarris,476256944803694179079458816,0


In [38]:
import folium

In [42]:
usmap = folium.Map(location=[39.8283, -98.5795],
                   zoom_start=4,
                   detect_retina=True)

In [43]:
#Create the choropleth and add it to the map

In [45]:
choropleth = folium.Choropleth(
    geo_data='us-states.json',              #File containing the GeoJSON that specifies the shapes to color
    name='choropleth',                      #Displays the Choropleth as a layer over the map.
    data=tweets_counts_by_state,            #DataFrame containing the values that determine the Choropleth colors
    columns=['State', 'Tweets'],            #List of two columns representing the keys and the corresponding values used to color the Choropleth
    key_on='feature.id',                    #Variable in the GeoJSON file to which the Choropleth binds the values in the columns argument
    fill_color='YlOrRd',                    #Color map specifying the colors to use to fill in the states
    fill_opacity=0.7,                       #A value from 0.0 (transparent) to 1.0 (opaque) of the fill colors displayed in the states
    line_opacity=0.2,                       #A value from 0.0 (transparent) to 1.0 (opaque) of lines used to delineate the states
    legend_name='Tweets by State'           #Indicate what the colors represent
).add_to(usmap)
layer = folium.LayerControl().add_to(usmap)

In [48]:
sorted_df = tweet_counts_df.sort_values(by='Tweets', ascending=False)
sorted_df.head()

,State,Name,Party,TwitterHandle,TwitterID,Tweets
0,AL,Richard Shelby,R,SenShelby,21111098,0
1,AL,Doug Jomes,D,SenDougJones,941080085121175552,0
2,AK,Lisa Murkowski,R,lisamurkowski,18061669,0
3,AK,Dan Sullivan,R,SenDanSullivan,2891210047,0
4,AZ,Jon Kyl,R,SenJonKyl,24905240,0


In [55]:
for index, (name, group) in enumerate(sorted_df.groupby('State')):      #Groups DF by State, maintaining the original row order in each group
    strings = [state_codes[name]]           #Used to assemble popup text. Look up the full state name in the state_codes dictionary
    for s in group.itertuples():            #Given senator's data
        strings.append(f'{s.Name} ({s.Party}); Tweets: {s.Tweets} ')
    text = '<br>'.join(strings)             #Use HTML for formatting
    marker = folium.Marker(
        (locations[index].latitude, locations[index].longitude),        #Tuple with the latitude and longitude
        popup=text)                 #Text to display if the user clicks the Marker
    marker.add_to(usmap)            #Add the Market to the map


[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


In [56]:
usmap.save('SenatorTweets.html')